# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [1]:
!pip install -U tokenizers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [3]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [4]:
# A smaller model that fits on a free Colab GPU
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# 2. Hardware optimization (Quantization)

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    #weights are stored in 4-bit -> running bigger models on smaller GPUs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 3. Load the tokenizer & model

In [6]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True # Does your model need it?
)
# mostly the outside models needs it
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?
#GPT/Llama/Mistral (decoder-only) - Left
#BERT/RoBERTa (encoder) - Right

Loading Tokenizer for mistralai/Mistral-7B-Instruct-v0.3...


In [7]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    dtype=torch.float16
)

Loading Model on Colab T4 GPU...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [9]:
prompt1 = [
    {
        "role": "system",
        "content": "You are a helpful software assistant. Your job is to explain the functionality of the provided code in simple terms."
    },
     {  "role": "user",
        "content":"""You are helping evaluate architecture recovery clustering results for Apache Lucene Codecs.The clustering algorithms compared are WCA, LIMBO, and ACDC.

A2A measures architecture-to-architecture distance/similarity between two clustering outputs.
CVG measures directional coverage from one clustering output to another.

Here are the results:

A2A WCA vs LIMBO: 82.94836956521739
CVG WCA vs LIMBO: 0.0, 0.0
A2A WCA vs ACDC: 82.20042046250876
CVG WCA vs ACDC: 0.0, 0.0
A2A LIMBO vs ACDC: 80.65872459705676
CVG LIMBO vs ACDC: 0.0, 0.0

Task:
1. Compare the clustering algorithms.
2. Identify which pair appears most similar.
3. Explain what high or low CVG values mean.
4. Write a short paragraph suitable for a Week 2 assignment report."""}
]

**Convert the prompt into tokens and Convert the output back from tokens into human-readable text and display it**

In [10]:
inputs1 = tokenizer.apply_chat_template(
    prompt1,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

In [11]:
# ── Define all parameter configs to compare ──────────────────────────────────
configs = [
    {
        "label": "Default (temp=0.5, top_p=0.8, sampling=True)",
        "max_new_tokens": 512,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
    },
    {
        "label": "Greedy (do_sample=False)",
        "max_new_tokens": 512,
        "do_sample": False,          # temperature & top_p are ignored here
    },
    {
        "label": "High creativity (temp=1.2, top_p=0.95)",
        "max_new_tokens": 512,
        "temperature": 1.2,
        "top_p": 0.95,
        "do_sample": True,
    },
    {
        "label": "Low creativity (temp=0.1, top_p=0.5)",
        "max_new_tokens": 512,
        "temperature": 0.1,
        "top_p": 0.5,
        "do_sample": True,
    },
]

# ── Generate and collect all outputs ─────────────────────────────────────────
results = []
input_length = inputs1['input_ids'].shape[1]

for cfg in configs:
    label = cfg.pop("label")           # remove label before passing to generate()
    print(f"Generating: {label} ...")

    outputs = model.generate(
        **inputs1,
        pad_token_id=tokenizer.eos_token_id,
        **cfg                          # unpack whichever params are in this config
    )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    results.append({"label": label, "config": cfg, "response": response})

    cfg["label"] = label               # restore label for readability later

# ── Print side-by-side comparison ────────────────────────────────────────────
separator = "=" * 70

for r in results:
    print(separator)
    print(f"CONFIG : {r['label']}")
    print(f"PARAMS : {r['config']}")
    print(separator)
    print(r['response'])
    print()

Generating: Default (temp=0.5, top_p=0.8, sampling=True) ...
Generating: Greedy (do_sample=False) ...
Generating: High creativity (temp=1.2, top_p=0.95) ...
Generating: Low creativity (temp=0.1, top_p=0.5) ...
CONFIG : Default (temp=0.5, top_p=0.8, sampling=True)
PARAMS : {'max_new_tokens': 512, 'temperature': 0.5, 'top_p': 0.8, 'do_sample': True, 'label': 'Default (temp=0.5, top_p=0.8, sampling=True)'}
1. Comparing the clustering algorithms: From the results provided, it appears that all three algorithms (WCA, LIMBO, and ACDC) are used to cluster data, but they produce different results. The A2A (Architecture-to-Architecture) values show the distance/similarity between the clustering outputs of each algorithm. The lower the A2A value, the more similar the clustering outputs are.

2. Identifying the pair that appears most similar: Based on the A2A values, the pair of clustering algorithms that appears most similar is WCA and LIMBO, as their A2A value is the lowest (82.94836956521739).


In [12]:
# Experiment 2: vary max_new_tokens (keep everything else fixed)
length_configs = [
    {
        "label": "Very short (64 tokens)",
        "max_new_tokens": 64,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
    },
    {
        "label": "Short (128 tokens)",
        "max_new_tokens": 128,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
    },
    {
        "label": "Medium (256 tokens)",
        "max_new_tokens": 256,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
    },
    {
        "label": "Long (512 tokens)",
        "max_new_tokens": 512,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
    },
]

# ── Generate and collect all outputs ─────────────────────────────────────────
results = []
input_length = inputs1['input_ids'].shape[1]

for cfg in length_configs:
    label = cfg.pop("label")           # remove label before passing to generate()
    print(f"Generating: {label} ...")

    outputs2 = model.generate(
        **inputs1,
        pad_token_id=tokenizer.eos_token_id,
        **cfg                          # unpack whichever params are in this config
    )

    response = tokenizer.decode(outputs2[0][input_length:], skip_special_tokens=True)
    results.append({"label": label, "config": cfg, "response": response})

    cfg["label"] = label               # restore label for readability later

# ── Print side-by-side comparison ────────────────────────────────────────────
separator = "=" * 70

for r in results:
    print(separator)
    print(f"CONFIG : {r['label']}")
    print(f"PARAMS : {r['config']}")
    print(separator)
    print(r['response'])
    print()

Generating: Very short (64 tokens) ...
Generating: Short (128 tokens) ...
Generating: Medium (256 tokens) ...
Generating: Long (512 tokens) ...
CONFIG : Very short (64 tokens)
PARAMS : {'max_new_tokens': 64, 'temperature': 0.5, 'top_p': 0.8, 'do_sample': True, 'label': 'Very short (64 tokens)'}
1. Comparing the clustering algorithms, we can see that WCA, LIMBO, and ACDC have been used to cluster the same data. The A2A (Architecture-to-Architecture) values indicate the distance or dissimilarity between the clusters produced by

CONFIG : Short (128 tokens)
PARAMS : {'max_new_tokens': 128, 'temperature': 0.5, 'top_p': 0.8, 'do_sample': True, 'label': 'Short (128 tokens)'}
1. Comparing the clustering algorithms: From the results provided, we can see that the three algorithms compared are WCA, LIMBO, and ACDC. The A2A (Architecture-to-Architecture) values indicate the distance or similarity between the clustering outputs of two algorithms. A lower A2A value suggests a higher similarity betw

In [13]:
# Experiment 3
repetition_configs = [

    {
        "label": "No repetition penalty",
        "max_new_tokens": 256,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
        "repetition_penalty": 1.0,   # default — no penalty
    },
    {
        "label": "Mild repetition penalty (1.2)",
        "max_new_tokens": 256,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
        "repetition_penalty": 1.2,
    },
    {
        "label": "Strong repetition penalty (1.5)",
        "max_new_tokens": 256,
        "temperature": 0.5,
        "top_p": 0.8,
        "do_sample": True,
        "repetition_penalty": 1.5,
    },
]

# ── Generate and collect all outputs ─────────────────────────────────────────
results = []
input_length = inputs1['input_ids'].shape[1]

for cfg in repetition_configs:
    label = cfg.pop("label")           # remove label before passing to generate()
    print(f"Generating: {label} ...")

    outputs3 = model.generate(
        **inputs1,
        pad_token_id=tokenizer.eos_token_id,
        **cfg                          # unpack whichever params are in this config
    )

    response = tokenizer.decode(outputs3[0][input_length:], skip_special_tokens=True)
    results.append({"label": label, "config": cfg, "response": response})

    cfg["label"] = label               # restore label for readability later

# ── Print side-by-side comparison ────────────────────────────────────────────
separator = "=" * 70

for r in results:
    print(separator)
    print(f"CONFIG : {r['label']}")
    print(f"PARAMS : {r['config']}")
    print(separator)
    print(r['response'])
    print()

Generating: No repetition penalty ...
Generating: Mild repetition penalty (1.2) ...
Generating: Strong repetition penalty (1.5) ...
CONFIG : No repetition penalty
PARAMS : {'max_new_tokens': 256, 'temperature': 0.5, 'top_p': 0.8, 'do_sample': True, 'repetition_penalty': 1.0, 'label': 'No repetition penalty'}
1. Comparing the clustering algorithms: From the results provided, we can see that the three algorithms compared (WCA, LIMBO, and ACDC) are used to cluster data in Apache Lucene Codecs. The A2A (Architecture-to-Architecture) measurements indicate the distance or similarity between the clustering outputs of each algorithm. The lower the A2A value, the more similar the clustering outputs are.

2. Identifying the pair that appears most similar: Based on the A2A values, the WCA and ACDC algorithms seem to produce the most similar clustering outputs, with a distance of 82.94836956521739 for WCA vs LIMBO and 82.20042046250876 for WCA vs ACDC.

3. Explanation of high or low CVG values: Th

In [15]:
# Experiment 4
beam_configs = [
    {
        "label": "Greedy (beam=1, no sample)",
        "max_new_tokens": 256,
        "do_sample": False,
        "num_beams": 1,
    },
    {
        "label": "Beam search (beam=4)",
        "max_new_tokens": 256,
        "do_sample": False,   # beam search doesn't use sampling
        "num_beams": 4,
    },
    {
        "label": "Beam search (beam=8)",
        "max_new_tokens": 256,
        "do_sample": False,
        "num_beams": 8,
        "length_penalty": 1.2,
    },
]


 #── Generate and collect all outputs ─────────────────────────────────────────
results = []
input_length = inputs1['input_ids'].shape[1]

for cfg in beam_configs:
    label = cfg.pop("label")           # remove label before passing to generate()
    print(f"Generating: {label} ...")

    outputs4 = model.generate(
        **inputs1,
        pad_token_id=tokenizer.eos_token_id,
        **cfg                          # unpack whichever params are in this config
    )

    response = tokenizer.decode(outputs4[0][input_length:], skip_special_tokens=True)
    results.append({"label": label, "config": cfg, "response": response})

    cfg["label"] = label               # restore label for readability later

# ── Print side-by-side comparison ────────────────────────────────────────────
separator = "=" * 70

for r in results:
    print(separator)
    print(f"CONFIG : {r['label']}")
    print(f"PARAMS : {r['config']}")
    print(separator)
    print(r['response'])
    print()

Generating: Greedy (beam=1, no sample) ...
Generating: Beam search (beam=4) ...
Generating: Beam search (beam=8) ...
CONFIG : Greedy (beam=1, no sample)
PARAMS : {'max_new_tokens': 256, 'do_sample': False, 'num_beams': 1, 'label': 'Greedy (beam=1, no sample)'}
1. Comparing the clustering algorithms, we can see that WCA, LIMBO, and ACDC have different performance characteristics. The A2A (Architecture-to-Architecture) values indicate the distance or similarity between the clustering outputs of each algorithm. A lower A2A value suggests a higher similarity between the clustering results. In this case, WCA and ACDC have a slightly lower A2A value compared to LIMBO, suggesting they are more similar to each other than either is to LIMBO.

2. The pair that appears most similar based on the provided results is WCA and ACDC, as they have the lowest A2A value (82.2004 for WCA vs ACDC compared to 82.9483 for WCA vs LIMBO and 80.6587 for LIMBO vs ACDC).

3. CVG (Coverage Vector Graph) values repr